# Advanced object-oriented programming 🧠

## What you will learn in this course 🧐🧐

You understand classes, objects, attributes, and methods from the fundamentals lecture. These concepts work well for simple systems. But real operations demand more sophisticated patterns. Real systems need **hierarchies** of related types, **uniform interfaces** for different implementations, and flexible ways to **combine** components.

By the end of this course, you will be able to:

- Build inheritance hierarchies for specialized types
- Implement polymorphism for uniform interfaces across different objects
- Use composition to build complex systems from simple parts
- Apply abstraction to hide implementation complexity
- Integrate objects with Python's built-in operations through special methods

In [1]:
from datetime import date
from abc import ABC, abstractmethod

## Building ProPublica's content management system

<img src="https://guides.propublica.org/design/assets/images/preview-main-wordmark.png" />

Your basic OOP skills created a working object management system. Now imagine you are the **lead developer at ProPublica**: the independent, non-profit investigative newsroom founded in 2008. ProPublica is funded entirely by foundations and reader donations. There is no advertising.

<Note title="Notable reporting and projects" type="info">

ProPublica has published many impactful investigations. "An Unbelievable Story of Rape" won the Pulitzer Prize for Explanatory Reporting in 2016 and was adapted into the Netflix series "Unbelievable". The "Dollars for Docs" database tracks payments from pharmaceutical companies to doctors, exposing conflicts of interest in healthcare. The "Electionland" project monitored voting issues during the 2016 US elections. The "Documenting Hate" initiative collects data on hate crimes and bias incidents across the US.

These projects show ProPublica's commitment to investigative journalism that holds power accountable and informs the public.

</Note>

ProPublica publishes two very different types of content that must coexist in the same system:

- **Investigations**: long-form, source-verified reporting by staff journalists. These require FOIA requests (public record requests to government agencies), a minimum number of documented sources, a right-of-reply sent to all named subjects, and often a coordinated **embargo** (an agreed hold on the publication date) with a partner outlet like NYT or Washington Post.
- **Data Reports**: data-driven pieces built on public datasets and scrapers. These go through a lighter editorial process but require a methodology statement and at least one peer review of the analysis before publication.

Both share common operations: *editorial review, status tracking, archiving*. But they follow completely different publication rules. The procedural approach would duplicate review and access logic across every content type. Basic OOP would create independent classes that still duplicate shared behavior.

In this lecture, you will learn how advanced OOP patterns solve these problems through **inheritance**, **polymorphism**, and **composition**.

## Abstraction and inheritance

<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Abstraction.png"/>

### Abstraction

**Abstraction = focus on *what* an object can do, not *how* it does it.**

You define a common interface. For example:

> Any `Content` at ProPublica can be reviewed, archived, and published.

You do not care whether it is a two-year investigation or a data report. From the outside, you just call these methods. The internal rules stay hidden inside each class.

### Inheritance

**Inheritance = reuse + specialization.**

- You create a **base class** (for example `Content`) with shared attributes and methods.
- Then you create **child classes** (for example `Investigation`, `DataReport`) that:
  - inherit everything from `Content`,
  - add or override behavior specific to their own editorial rules.

So you avoid rewriting status tracking and review logic for every content type.

### Abstract base classes in Python

Python lets you formalize this idea with `ABC` (**Abstract Base Class**) and the `@abstractmethod` decorator.

You define a class that says:

> Any subclass of me **must** implement these methods.

It is literally an **editorial contract**: if you are a type of content on this platform, you promise to be publishable and to describe your format.

Here is the abstract `Content` class. It defines what every content type must have. It cannot be instantiated directly.

In [2]:
class Content(ABC):
    def __init__(self, title, author):
        self.title = title
        self.author = author
        self.status = 'draft'  # draft -> reviewed -> published

    def submit_for_review(self):
        # Shared logic: same for every content type
        self.status = 'reviewed'
        return f"'{self.title}' submitted for review"

    @abstractmethod
    def publish(self):
        # Each content type must define its own rules
        pass

- `class Content(ABC):` means `Content` inherits from `ABC` (Abstract Base Class).
- `submit_for_review` is a **concrete method**. It has real code and is shared by every subclass.
- `publish` is marked `@abstractmethod`. It has no real code here. Every child class must provide its own version, or Python will refuse to create objects from it.

<Note type="important">

An abstract class sets the **rules of membership**. It does not do the work itself. It forces every concrete child to implement the required methods.

</Note>

Now let's create a concrete child class. `Investigation` inherits from `Content` and adds its own strict publication rules.

In [3]:
class Investigation(Content):
    def __init__(self, title, author, beat):
        # Reuse Content's constructor
        super().__init__(title, author)
        # Add investigation-specific attributes
        self.beat = beat  # e.g. 'tax policy', 'criminal justice'
        self.sources = []
        self.right_of_reply_sent = False

    def add_source(self, description):
        self.sources.append(description)

    def send_right_of_reply(self):
        self.right_of_reply_sent = True

    def publish(self):
        # Investigation-specific rules
        if self.status != 'reviewed':
            return f"Cannot publish '{self.title}': review pending"
        if len(self.sources) < 2:
            return f"Cannot publish '{self.title}': need 2 sources"
        if not self.right_of_reply_sent:
            return f"Cannot publish '{self.title}': right of reply missing"
        self.status = 'published'
        return f"Investigation published: '{self.title}' [{self.beat}]"

- `class Investigation(Content):` creates a child class. The `(Content)` part means "inherit from Content".
- `super().__init__(title, author)` calls the **parent constructor**. This sets up `self.title`, `self.author`, and `self.status` without rewriting that code.
- The class adds its own attributes (`beat`, `sources`, `right_of_reply_sent`) and methods (`add_source`, `send_right_of_reply`).
- `publish` **implements** the abstract method. Each check protects the newsroom's editorial standards. No review, no publication. No sources, no publication.

Let's use it.

In [4]:
story = Investigation(
    title="How the IRS Fails to Audit Wealthy Americans",
    author="Jesse Eisinger",
    beat="tax policy"
)

story.submit_for_review()        # inherited from Content
story.add_source("FOIA response")  # defined in Investigation
story.add_source("Whistleblower testimony")
story.send_right_of_reply()

print(story.publish())

Investigation published: 'How the IRS Fails to Audit Wealthy Americans' [tax policy]


Notice the two kinds of methods being called.

- `submit_for_review()` is **inherited** from `Content`. It was written once and reused.
- `add_source()` and `send_right_of_reply()` are **specific** to `Investigation`.
- `publish()` is the **implementation** of the abstract method. It applies investigation rules.

> Inheritance gives you **reuse without duplication**. Shared logic lives once, in the parent. Specialization lives in each child.

## Polymorphism

**Polymorphism** lets you write code once that works with many types. The word means "many forms". One function call, many possible behaviors behind it.

For ProPublica, this is critical. The newsroom runs a daily publish pipeline. It should not care whether a piece is an investigation, a data report, or a future format. It just says "publish" and each content type applies its own rules.

Let's add a second subclass: `DataReport`. It has different publication rules.

In [5]:
class DataReport(Content):
    def __init__(self, title, author, tool="Python"):
        super().__init__(title, author)
        self.tool = tool
        self.datasets = []
        self.peer_reviews = 0

    def add_dataset(self, name):
        self.datasets.append(name)

    def add_peer_review(self):
        self.peer_reviews += 1

    def publish(self):
        # Data report rules: datasets and peer review required
        if self.status != 'reviewed':
            return f"Cannot publish '{self.title}': review pending"
        if len(self.datasets) == 0:
            return f"Cannot publish '{self.title}': no datasets"
        if self.peer_reviews < 1:
            return f"Cannot publish '{self.title}': peer review missing"
        self.status = 'published'
        return f"Data report published: '{self.title}' [{self.tool}]"

`DataReport` inherits from `Content` like `Investigation` did. But its `publish` method checks completely different rules. Same method name, different behavior. That is polymorphism at work.

Now we write **one pipeline function** that works with any `Content` object.

In [6]:
def daily_publish_run(pipeline):
    print("=== DAILY PUBLISH RUN ===")
    for content in pipeline:
        # Polymorphic call: each object applies its own rules
        result = content.publish()
        print(result)

`daily_publish_run` does not know or care which subclass it is handling. It just calls `.publish()`. Python picks the right version automatically, based on the object's actual type.

Let's build a mixed pipeline and run it.

In [7]:
# Prepare an investigation
story = Investigation("IRS Audit Gaps", "Jesse Eisinger", "tax policy")
story.submit_for_review()
story.add_source("FOIA response")
story.add_source("Whistleblower testimony")
story.send_right_of_reply()

# Prepare a data report
report = DataReport("Hospital Billing Disparities", "Lena Groeger")
report.submit_for_review()
report.add_dataset("CMS Hospital Price Transparency")
report.add_peer_review()

# Mix them in one pipeline
pipeline = [story, report]
daily_publish_run(pipeline)

=== DAILY PUBLISH RUN ===
Investigation published: 'IRS Audit Gaps' [tax policy]
Data report published: 'Hospital Billing Disparities' [Python]


The pipeline function called `.publish()` on both objects. Each responded with its own rules. The investigation checked sources and right of reply. The data report checked datasets and peer review.

<Note type="tip">

Polymorphism delivers **extensibility**. Tomorrow you can add a `Newsletter` or `Podcast` class. As long as it inherits from `Content` and implements `publish()`, `daily_publish_run` will handle it without a single change.

</Note>

> Editorial rules live inside each class. The pipeline code stays clean and generic.

## Composition

<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Composition.png"/>

**Composition** is a design principle. A class is built from **other classes**, not by inheriting from them. Instead of saying *"A is a kind of B"*, you say *"A has a B"*.

At ProPublica, related investigations are grouped into **Series**: long-running thematic collections. For example, all pieces covering the IRS and tax inequality live in one `Series`. A series **has a** lead journalist. A series **has** content pieces.

This is different from inheritance. A series is not a kind of journalist. A series is not a kind of content piece. A series is a container that holds them.

Here are the small building blocks.

In [8]:
class Journalist:
    def __init__(self, name, beat):
        self.name = name
        self.beat = beat


class ContentItem:
    def __init__(self, title, content_type):
        self.title = title
        self.content_type = content_type  # 'investigation', 'data report'

Now the `Series` class **composes** these two classes together.

In [9]:
class Series:
    def __init__(self, name, lead_journalist):
        self.name = name
        self.lead_journalist = lead_journalist  # HAS a journalist
        self.pieces = []                         # HAS content pieces

    def add_piece(self, piece):
        self.pieces.append(piece)

`Series` does not inherit from `Journalist` or `ContentItem`. It **contains** them. A series takes a journalist as an argument. A series holds a list of pieces. Each part keeps its own identity and its own behavior.

Let's build a real example: ProPublica's "The Secret IRS Files" series.

In [10]:
# Build the lead journalist
jesse = Journalist(name="Jesse Eisinger", beat="tax policy")

# Build the series with the journalist
secret_irs_files = Series(
    name="The Secret IRS Files",
    lead_journalist=jesse
)

# Add content pieces
piece1 = ContentItem("The Secret IRS Files", "investigation")
piece2 = ContentItem("Billionaires Paid Little in Taxes", "investigation")
piece3 = ContentItem("FAQ About the Tax Data", "explainer")

secret_irs_files.add_piece(piece1)
secret_irs_files.add_piece(piece2)
secret_irs_files.add_piece(piece3)

print(f"Series: {secret_irs_files.name}")
print(f"Lead: {secret_irs_files.lead_journalist.name}")
print(f"Pieces: {len(secret_irs_files.pieces)}")

Series: The Secret IRS Files
Lead: Jesse Eisinger
Pieces: 3


`secret_irs_files.lead_journalist.name` shows the power of composition. The `Series` holds a `Journalist` object. You reach into it to read the journalist's name. Each object keeps its own behavior.

<Note type="tip">

- **Composition** = "**has-a**"
  - A `Series` **has a** lead journalist
  - A `Series` **has** content pieces

- **Inheritance** = "**is-a**"
  - An `Investigation` **is a** `Content`
  - A `DataReport` **is a** `Content`

Composition is great when:
- You assemble systems from interchangeable parts.
- You want to **swap, add, or remove** pieces at any time.
- You want each part to evolve on its own.

</Note>

> A good rule of thumb: **prefer composition to inheritance** when parts should be replaceable or when the "is-a" relationship feels forced.

## Resources 📚📚

- [The ProPublica website](https://www.propublica.org/) 
- [Real Python's guide on Inheritance vs Composition](https://realpython.com/python3-object-oriented-programming/) 
- [The official Python documentation on `abc`](https://docs.python.org/3/library/abc.html) 
- [The Python data model page](https://docs.python.org/3/reference/datamodel.html#special-method-names)
- [SOLID principles](https://en.wikipedia.org/wiki/SOLID) 